<a href="https://colab.research.google.com/github/kuds/rl-doom/blob/main/notebooks/02_dqn_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — DQN Training

Train a **Double DQN** agent on the ViZDoom *Basic* scenario. We log learning
curves (reward, loss, epsilon) and render the trained agent's gameplay.

## 1. Setup

In [ ]:
# --- Colab Setup ---
# Uncomment the block below when running on Google Colab
# import subprocess, os
# if not os.path.exists("/content/rl-doom"):
#     subprocess.run(["git", "clone", "https://github.com/kuds/rl-doom.git", "/content/rl-doom"], check=True)
# os.chdir("/content/rl-doom/notebooks")
# subprocess.run(["pip", "install", "-q", "-e", "/content/rl-doom[notebooks]"], check=True)

import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt
import torch
from collections import deque

from rl_doom.env import DoomEnv, ResizeObservation, SkipFrame, FrameStack
from rl_doom.models import DQNNetwork
from rl_doom.agents.dqn import DQNAgent
from rl_doom.replay_buffer import ReplayBuffer
from rl_doom.paths import (
    new_run_dir, write_config, mark_run_status, update_latest_symlink,
    new_sweep_dir, new_sweep_variant_dir,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the block below when running on Google Colab
# ---
# import shutil
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ROOT = "/content/drive/MyDrive/Finding Theta/rl-doom"
# os.makedirs(DRIVE_ROOT, exist_ok=True)
# for subdir in ["training_jobs", "analysis"]:
#     drive_dir = f"{DRIVE_ROOT}/{subdir}"
#     local_dir = os.path.abspath(f"../{subdir}")
#     os.makedirs(drive_dir, exist_ok=True)
#     if os.path.islink(local_dir):
#         os.remove(local_dir)
#     if os.path.isdir(local_dir):
#         for f in os.listdir(local_dir):
#             src = os.path.join(local_dir, f)
#             dst = os.path.join(drive_dir, f)
#             if not os.path.exists(dst):
#                 shutil.move(src, dst)
#         shutil.rmtree(local_dir)
#     os.symlink(drive_dir, local_dir)
# print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")
# ---

## 2. Environment

In [ ]:
def make_env(scenario="basic", seed=42):
    env = DoomEnv(scenario=scenario)
    env = ResizeObservation(env, shape=(84, 84))
    env = SkipFrame(env, skip=4)
    env = FrameStack(env, num_stack=4)
    return env

env = make_env()
n_actions = env.action_space.n
obs, _ = env.reset(seed=42)
obs_shape = obs.shape  # Store shape before env gets closed later
print(f"Observation shape: {obs_shape}, Actions: {n_actions}")

## 3. Hyperparameters

In [ ]:
config = dict(
    # Training
    total_steps=100_000,
    learning_starts=1_000,
    train_freq=4,
    batch_size=32,
    # DQN
    lr=1e-4,
    gamma=0.99,
    target_update_freq=1_000,
    # Exploration
    eps_start=1.0,
    eps_end=0.01,
    eps_decay_steps=50_000,
    # Replay
    buffer_size=50_000,
    # Logging
    log_freq=500,
    eval_freq=5_000,
    eval_episodes=10,
)
config

## 4. Initialize agent & replay buffer

In [ ]:
agent = DQNAgent(
    obs_shape=obs_shape,
    n_actions=n_actions,
    lr=config["lr"],
    gamma=config["gamma"],
    device=device,
)

replay_buffer = ReplayBuffer(capacity=config["buffer_size"])

print(f"Network parameters: {sum(p.numel() for p in agent.policy_net.parameters()):,}")

## 5. Training loop

In [ ]:
import time
from torch.utils.tensorboard import SummaryWriter

def linear_schedule(step, start, end, duration):
    """Linearly anneal from start to end over duration steps."""
    frac = min(step / duration, 1.0)
    return start + frac * (end - start)


# Create the run directory for this training job. All checkpoints, metrics,
# tensorboard logs, figures, and media for this run land under run_dir.
SEED = 42
run_dir = new_run_dir("basic", "dqn", seed=SEED)
write_config(run_dir, env="basic", algo="dqn", seed=SEED, hyperparams=config)
print(f"Run dir: {run_dir}")

writer = SummaryWriter(log_dir=str(run_dir / "tensorboard"))

# Logging containers
episode_rewards_log = []
episode_lengths_log = []
losses_log = []
epsilons_log = []
eval_rewards_log = []
q_value_log = []          # mean Q-value at log checkpoints
action_counts = np.zeros(n_actions, dtype=np.int64)

obs, _ = env.reset(seed=SEED)
episode_reward = 0.0
episode_steps = 0
episode_count = 0
recent_rewards = deque(maxlen=20)
t_start = time.time()
total_env_steps = 0

for step in range(1, config["total_steps"] + 1):
    # Epsilon schedule
    epsilon = linear_schedule(
        step, config["eps_start"], config["eps_end"], config["eps_decay_steps"]
    )

    # Select action
    action = agent.select_action(obs, epsilon=epsilon)
    action_counts[action] += 1

    # Step environment
    next_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    replay_buffer.push(obs, action, reward, next_obs, done)
    obs = next_obs
    episode_reward += reward
    episode_steps += 1
    total_env_steps += 1

    # Episode bookkeeping
    if done:
        episode_count += 1
        episode_rewards_log.append(episode_reward)
        episode_lengths_log.append(episode_steps)
        recent_rewards.append(episode_reward)
        writer.add_scalar("episode/reward", episode_reward, episode_count)
        writer.add_scalar("episode/length", episode_steps, episode_count)
        obs, _ = env.reset()
        episode_reward = 0.0
        episode_steps = 0

    # Train
    if step >= config["learning_starts"] and step % config["train_freq"] == 0:
        batch = replay_buffer.sample(config["batch_size"])
        loss = agent.train_step(batch)
        losses_log.append(loss)
        writer.add_scalar("train/loss", loss, step)

    # Target network update
    if step % config["target_update_freq"] == 0:
        agent.update_target()

    # Logging
    if step % config["log_freq"] == 0:
        epsilons_log.append(epsilon)
        writer.add_scalar("train/epsilon", epsilon, step)

        # Log mean Q-value over a small sample
        if len(replay_buffer) >= config["batch_size"]:
            with torch.no_grad():
                q_batch = replay_buffer.sample(config["batch_size"])
                q_obs = torch.FloatTensor(np.array(q_batch["obs"])).to(device)
                q_vals = agent.policy_net(q_obs)
                mean_q = q_vals.max(dim=1).values.mean().item()
                q_value_log.append(mean_q)
                writer.add_scalar("train/mean_q_value", mean_q, step)

        avg = np.mean(recent_rewards) if recent_rewards else 0
        elapsed = time.time() - t_start
        fps = total_env_steps / elapsed if elapsed > 0 else 0
        print(
            f"Step {step:>7,} | Eps {epsilon:.3f} | "
            f"Episodes {episode_count} | Avg reward (20) {avg:.2f} | "
            f"FPS {fps:.0f}"
        )

    # Periodic evaluation
    if step % config["eval_freq"] == 0:
        eval_env = make_env()
        eval_rews = []
        eval_lens = []
        for _ in range(config["eval_episodes"]):
            eo, _ = eval_env.reset()
            er, el = 0.0, 0
            d = False
            while not d:
                a = agent.select_action(eo, epsilon=0.0)
                eo, r, term, trunc, _ = eval_env.step(a)
                er += r
                el += 1
                d = term or trunc
            eval_rews.append(er)
            eval_lens.append(el)
        eval_env.close()
        eval_rewards_log.append((step, np.mean(eval_rews), np.std(eval_rews),
                                 np.mean(eval_lens), np.std(eval_lens)))
        writer.add_scalar("eval/mean_reward", np.mean(eval_rews), step)
        writer.add_scalar("eval/mean_length", np.mean(eval_lens), step)
        print(f"  >> Eval: {np.mean(eval_rews):.2f} ± {np.std(eval_rews):.2f}")

wall_time = time.time() - t_start
env.close()
writer.close()
print(f"\nTraining complete! Wall time: {wall_time:.1f}s | Avg FPS: {total_env_steps / wall_time:.0f}")

## 6. Learning curves

In [ ]:
import platform, datetime

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# -- Episode rewards --
ax = axes[0, 0]
ax.plot(episode_rewards_log, alpha=0.4, label="Raw")
if len(episode_rewards_log) >= 20:
    smoothed = np.convolve(episode_rewards_log, np.ones(20)/20, mode="valid")
    ax.plot(range(19, 19 + len(smoothed)), smoothed, label="MA-20")
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.set_title("Episode Rewards")
ax.legend()

# -- Episode lengths --
ax = axes[0, 1]
ax.plot(episode_lengths_log, alpha=0.4, color="green", label="Raw")
if len(episode_lengths_log) >= 20:
    smoothed = np.convolve(episode_lengths_log, np.ones(20)/20, mode="valid")
    ax.plot(range(19, 19 + len(smoothed)), smoothed, color="darkgreen", label="MA-20")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps")
ax.set_title("Episode Lengths")
ax.legend()

# -- Loss --
ax = axes[0, 2]
ax.plot(losses_log, alpha=0.3)
if len(losses_log) >= 100:
    smoothed = np.convolve(losses_log, np.ones(100)/100, mode="valid")
    ax.plot(range(99, 99 + len(smoothed)), smoothed, color="red")
ax.set_xlabel("Training step")
ax.set_ylabel("Huber Loss")
ax.set_title("Training Loss")

# -- Epsilon --
ax = axes[1, 0]
ax.plot(epsilons_log)
ax.set_xlabel(f"Log step (x{config['log_freq']})")
ax.set_ylabel("Epsilon")
ax.set_title("Exploration Schedule")

# -- Mean Q-values --
ax = axes[1, 1]
if q_value_log:
    ax.plot(q_value_log)
    ax.set_xlabel(f"Log step (x{config['log_freq']})")
    ax.set_ylabel("Mean Max Q")
    ax.set_title("Mean Q-Value")
else:
    ax.text(0.5, 0.5, "No Q-value data", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Mean Q-Value")

# -- Action distribution --
ax = axes[1, 2]
ax.bar(range(n_actions), action_counts)
ax.set_xlabel("Action")
ax.set_ylabel("Count")
ax.set_title("Action Distribution (Training)")
ax.set_xticks(range(n_actions))

plt.tight_layout()
plt.savefig(run_dir / "figures" / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Persist training logs with reproducibility metadata
np.savez(
    run_dir / "metrics" / "training.npz",
    episode_rewards=np.array(episode_rewards_log),
    episode_lengths=np.array(episode_lengths_log),
    losses=np.array(losses_log),
    epsilons=np.array(epsilons_log),
    eval_rewards=np.array(eval_rewards_log) if eval_rewards_log else np.array([]),
    q_values=np.array(q_value_log),
    action_counts=action_counts,
    # Reproducibility metadata
    config=str(config),
    seed=SEED,
    wall_time_seconds=wall_time,
    fps=total_env_steps / wall_time if wall_time > 0 else 0,
    total_env_steps=total_env_steps,
    timestamp=str(datetime.datetime.now(datetime.timezone.utc)),
    python_version=platform.python_version(),
    platform_info=platform.platform(),
    torch_version=torch.__version__,
    device=str(device),
)

## 7. Evaluation rewards over training

In [ ]:
if eval_rewards_log:
    eval_arr = np.array(eval_rewards_log)
    steps = eval_arr[:, 0]
    means = eval_arr[:, 1]
    stds = eval_arr[:, 2]
    mean_lens = eval_arr[:, 3]
    std_lens = eval_arr[:, 4]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Eval rewards
    ax = axes[0]
    ax.plot(steps, means, marker="o")
    ax.fill_between(steps, means - stds, means + stds, alpha=0.2)
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Eval Reward")
    ax.set_title("DQN — Evaluation Reward (Basic)")
    ax.grid(True, alpha=0.3)

    # Eval episode lengths
    ax = axes[1]
    ax.plot(steps, mean_lens, marker="o", color="green")
    ax.fill_between(steps, mean_lens - std_lens, mean_lens + std_lens, alpha=0.2, color="green")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Eval Episode Length")
    ax.set_title("DQN — Evaluation Episode Length (Basic)")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(run_dir / "figures" / "eval_performance.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Hyperparameter sensitivity

Quick exploration: sweep over learning rates to see the effect on final performance.

In [ ]:
# NOTE: This cell is expensive — set QUICK_SWEEP_STEPS to a small value for a
# fast smoke test, or skip this cell entirely.
import csv, yaml, datetime as _dt

QUICK_SWEEP_STEPS = 20_000
LR_CANDIDATES = [3e-4, 1e-4, 3e-5]

# Create a sweep directory with a date-stamped name so repeated sweeps don't clash.
sweep_name = f"lr_{_dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%d')}"
sweep_dir = new_sweep_dir("basic", "dqn", sweep_name)

# Freeze sweep definition up front so the intent is self-documenting.
sweep_def = {
    "name": sweep_name,
    "algo": "dqn",
    "env": "basic",
    "base_config": {
        "total_steps": QUICK_SWEEP_STEPS,
        "gamma": 0.99,
        "buffer_size": config["buffer_size"],
        "batch_size": 32,
        "train_freq": 4,
        "target_update_freq": 1_000,
    },
    "grid": {"lr": LR_CANDIDATES},
    "intent": "Quick LR sensitivity sweep for DQN on Basic",
}
(sweep_dir / "sweep.yaml").write_text(yaml.safe_dump(sweep_def, sort_keys=False))

sweep_results = {}
summary_rows = []

for lr in LR_CANDIDATES:
    variant_name = f"lr_{lr:.0e}"
    variant_dir = new_sweep_variant_dir(sweep_dir, variant_name)
    variant_writer = SummaryWriter(log_dir=str(variant_dir / "tensorboard"))

    sweep_env = make_env()
    sweep_obs, _ = sweep_env.reset(seed=42)
    sweep_agent = DQNAgent(
        obs_shape=obs_shape, n_actions=n_actions, lr=lr, gamma=0.99, device=device
    )
    sweep_buf = ReplayBuffer(capacity=config["buffer_size"])

    ep_rews, ep_r = [], 0.0
    t_variant = time.time()

    for s in range(1, QUICK_SWEEP_STEPS + 1):
        eps = linear_schedule(s, 1.0, 0.01, QUICK_SWEEP_STEPS // 2)
        a = sweep_agent.select_action(sweep_obs, epsilon=eps)
        ns, r, term, trunc, _ = sweep_env.step(a)
        sweep_buf.push(sweep_obs, a, r, ns, term or trunc)
        sweep_obs, ep_r = ns, ep_r + r
        if term or trunc:
            ep_rews.append(ep_r)
            variant_writer.add_scalar("episode/reward", ep_r, len(ep_rews))
            sweep_obs, _ = sweep_env.reset()
            ep_r = 0.0
        if s >= 1_000 and s % 4 == 0:
            sweep_agent.train_step(sweep_buf.sample(32))
        if s % 1_000 == 0:
            sweep_agent.update_target()

    sweep_env.close()
    variant_writer.close()

    variant_wall = time.time() - t_variant
    final_mean = float(np.mean(ep_rews[-20:])) if ep_rews else float("nan")
    final_std = float(np.std(ep_rews[-20:])) if ep_rews else float("nan")

    # Persist per-variant metrics + weights
    np.savez(
        variant_dir / "metrics" / "training.npz",
        episode_rewards=np.array(ep_rews),
        lr=lr,
        total_steps=QUICK_SWEEP_STEPS,
        wall_time_seconds=variant_wall,
    )
    sweep_agent.save(str(variant_dir / "checkpoints" / "final.pt"))

    sweep_results[lr] = ep_rews
    summary_rows.append({
        "variant": variant_name,
        "lr": lr,
        "n_episodes": len(ep_rews),
        "final_eval_mean": final_mean,
        "final_eval_std": final_std,
        "wall_time_seconds": variant_wall,
        "variant_dir": variant_name,
    })
    print(f"lr={lr:.0e}  episodes={len(ep_rews)}  mean={final_mean:.2f}  wall={variant_wall:.1f}s")

# Write sweep summary table
with open(sweep_dir / "summary.csv", "w", newline="") as f:
    writer_csv = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer_csv.writeheader()
    writer_csv.writerows(summary_rows)

# Plot sensitivity curves (save figure inside the sweep dir for colocation)
plt.figure(figsize=(10, 5))
for lr, rews in sweep_results.items():
    if len(rews) >= 10:
        sm = np.convolve(rews, np.ones(10)/10, mode="valid")
        plt.plot(sm, label=f"lr={lr:.0e}")
plt.xlabel("Episode")
plt.ylabel("Reward (MA-10)")
plt.title("LR Sensitivity — DQN Basic")
plt.legend()
plt.tight_layout()
plt.savefig(sweep_dir / "sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Sweep artifacts saved to {sweep_dir}")

## 9. Render trained agent

In [ ]:
from rl_doom.evaluate import record_episode

frames = record_episode(agent, make_env, epsilon=0.0)
print(f"Recorded {len(frames)} frames")

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(6, 6))
ax.axis("off")
img = ax.imshow(frames[0])

def update(i):
    img.set_data(frames[i])
    return [img]

anim = FuncAnimation(fig, update, frames=len(frames), interval=50, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

## 10. Save checkpoint

In [ ]:
# Save full checkpoint with metadata for resumable training
checkpoint = {
    "model_state_dict": agent.policy_net.state_dict(),
    "target_state_dict": agent.policy_net.state_dict(),  # target net weights
    "config": config,
    "training_step": config["total_steps"],
    "episode_count": episode_count,
    "epsilon": epsilon,
    "wall_time_seconds": wall_time,
    "total_env_steps": total_env_steps,
    "mean_eval_reward": eval_rewards_log[-1][1] if eval_rewards_log else None,
}
torch.save(checkpoint, run_dir / "checkpoints" / "final_full.pt")

# Also save lightweight weights-only checkpoint (backward compatible)
agent.save(str(run_dir / "checkpoints" / "final.pt"))

# Finalize: record status and update the "latest" symlink for notebook 04.
mark_run_status(
    run_dir,
    status="completed",
    wall_time_seconds=wall_time,
    total_env_steps=total_env_steps,
    mean_eval_reward=float(eval_rewards_log[-1][1]) if eval_rewards_log else None,
)
update_latest_symlink("basic", "dqn", run_dir)
print(f"Checkpoints saved (full + weights-only) to {run_dir / 'checkpoints'}")

In [ ]:
# Disconnect the Colab runtime at the end of the notebook to save compute.
# No-op when running locally.
try:
    from google.colab import runtime as _colab_runtime
except ImportError:
    pass
else:
    import time
    print("Notebook finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    _colab_runtime.unassign()